# Lab | Data Aggregation and Filtering

In this challenge, we will continue to work with customer data from an insurance company. We will use the dataset called marketing_customer_analysis.csv, which can be found at the following link:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis.csv

This dataset contains information such as customer demographics, policy details, vehicle information, and the customer's response to the last marketing campaign. Our goal is to explore and analyze this data by first performing data cleaning, formatting, and structuring.

1. Create a new DataFrame that only includes customers who:
   - have a **low total_claim_amount** (e.g., below $1,000),
   - have a response "Yes" to the last marketing campaign.

2. Using the original Dataframe, analyze:
   - the average `monthly_premium` and/or customer lifetime value by `policy_type` and `gender` for customers who responded "Yes", and
   - compare these insights to `total_claim_amount` patterns, and discuss which segments appear most profitable or low-risk for the company.

3. Analyze the total number of customers who have policies in each state, and then filter the results to only include states where there are more than 500 customers.

4. Find the maximum, minimum, and median customer lifetime value by education level and gender. Write your conclusions.

## Bonus

5. The marketing team wants to analyze the number of policies sold by state and month. Present the data in a table where the months are arranged as columns and the states are arranged as rows.

6.  Display a new DataFrame that contains the number of policies sold by month, by state, for the top 3 states with the highest number of policies sold.

*Hint:*
- *To accomplish this, you will first need to group the data by state and month, then count the number of policies sold for each group. Afterwards, you will need to sort the data by the count of policies sold in descending order.*
- *Next, you will select the top 3 states with the highest number of policies sold.*
- *Finally, you will create a new DataFrame that contains the number of policies sold by month for each of the top 3 states.*

7. The marketing team wants to analyze the effect of different marketing channels on the customer response rate.

Hint: You can use melt to unpivot the data and create a table that shows the customer response rate (those who responded "Yes") by marketing channel.

External Resources for Data Filtering: https://towardsdatascience.com/filtering-data-frames-in-pandas-b570b1f834b9

In [6]:
# your code goes here
#Step: 1

import pandas as pd

url = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis.csv"
df = pd.read_csv(url)

df.columns = (
    df.columns
        .str.strip() # remove leading/trailing spaces
        .str.lower()
        .str.replace(" ", "_")  # replace internal spaces with _
)
df.columns

df_filtered = df[(df["total_claim_amount"] < 1000) & (df["response"] == "Yes")]

df_filtered = df_filtered.reset_index(drop = True)
df_filtered.head()

,unnamed:_0,customer,state,customer_lifetime_value,response,coverage,education,effective_to_date,employmentstatus,gender,income,location_code,marital_status,monthly_premium_auto,months_since_last_claim,months_since_policy_inception,number_of_open_complaints,number_of_policies,policy_type,policy,renew_offer_type,sales_channel,total_claim_amount,vehicle_class,vehicle_size,vehicle_type
0,3,XL78013,Oregon,22332.439460,Yes,Extended,College,1/11/11,Employed,M,49078,Suburban,Single,97,10.0,3,0.0,2,Corporate Auto,Corporate L3,Offer2,Branch,484.013411,Four-Door Car,Medsize,A
1,8,FM55990,California,5989.773931,Yes,Premium,College,1/19/11,Employed,M,66839,Suburban,Single,154,33.0,24,0.0,1,Personal Auto,Personal L1,Offer2,Branch,739.200000,Sports Car,Medsize,NaN
2,15,CW49887,California,4626.801093,Yes,Basic,Master,1/16/11,Employed,F,79487,Suburban,Divorced,114,20.0,87,0.0,1,Special Auto,Special L1,Offer2,Branch,547.200000,SUV,Medsize,NaN
3,19,NJ54277,California,3746.751625,Yes,Extended,College,2/26/11,Employed,F,41479,Rural,Married,94,14.0,38,1.0,1,Personal Auto,Personal L2,Offer2,Call Center,19.575683,Two-Door Car,Large,A
4,27,MQ68407,Oregon,4376.363592,Yes,Premium,Bachelor,2/28/11,Employed,F,63774,Rural,Divorced,111,18.0,63,0.0,1,Personal Auto,Personal L3,Offer2,Agent,60.036683,Four-Door Car,Medsize,NaN


In [7]:
# Step: 2 

df_yes = df[df["response"] == "Yes"]

summary = df_yes.groupby(["policy_type", "gender"]).agg(
    avg_monthly_income = ("monthly_premium_auto", "mean"),
    avg_clv = ("customer_lifetime_value", "mean"),
    avg_total_claim = ("total_claim_amount", "mean"),
    count = ("customer", "count")
).reset_index()

summary

,policy_type,gender,avg_monthly_income,avg_clv,avg_total_claim,count
0,Corporate Auto,F,94.301775,7712.628736,433.738499,169
1,Corporate Auto,M,92.188312,7944.465414,408.582459,154
2,Personal Auto,F,98.998148,8339.791842,452.965929,540
3,Personal Auto,M,91.085821,7448.383281,457.010178,536
4,Special Auto,F,92.314286,7691.584111,453.280164,35
5,Special Auto,M,86.343750,8247.088702,429.527942,32


In [11]:
# Step 3: 

state_counts = df['state'].value_counts().reset_index()
state_counts.columns = ['state', 'customer_count']

states_over_500 = state_counts[state_counts['customer_count'] > 500]
states_over_500

,state,customer_count
0,California,3552
1,Oregon,2909
2,Arizona,1937
3,Nevada,993
4,Washington,888


In [10]:
# Step 4:

df["customer_lifetime_value"] = pd.to_numeric(df["customer_lifetime_value"], errors = "coerce")

clv_stats = df.groupby(["education", "gender"])["customer_lifetime_value"].agg(["min", "median", "max", "count"]).reset_index()
clv_stats

,education,gender,min,median,max,count
0,Bachelor,F,1904.000852,5640.505303,73225.95652,1700
1,Bachelor,M,1898.007675,5548.031892,67907.27050,1572
2,College,F,1898.683686,5623.611187,61850.18803,1608
3,College,M,1918.119700,6005.847375,61134.68307,1596
4,Doctor,F,2395.570000,5332.462694,44856.11397,200
5,Doctor,M,2267.604038,5577.669457,32677.34284,201
6,High School or Below,F,2144.921535,6039.553187,55277.44589,1590
7,High School or Below,M,1940.981221,6286.731006,83325.38119,1556
8,Master,F,2417.777032,5729.855012,51016.06704,475
9,Master,M,2272.307310,5579.099207,50568.25912,412
